<!-- dd:dd-lesson-np-3 -->

# Vectorization and broadcasting

*Numpy · `np-3`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "np-3"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrtHWuP2zbyrwgLHGDfWT4+9SjQf9FvjhE4G6ddYGNvbSeXpuh/P3L4lETZIiXLDe6C1ivrxZnhcF6cGf/5hDF++in78+nT"
    "Qfx5Oh9fv+6fVtnT8+68P4szmz+fzvvLl7f3z8ePe3nHy+e34+mSXY6n59+y3Tm7vDvssp+zy3p32h1+3S/ocn3an3/bvYnD"
    "VYaX7w4fei7jVUaXaqzXV/nqxevu84ePu+z0U7Z4Ob8czpfd4Xm/OK3E47/sD+fjaSkPd+f3F/i2OC3Xl+Pry/myWC6XC4B9"
    "sVtlH5bw2v23t/3zZf/xvTg4wft/OX3Zr7LNBgnAVhnZimP5V8AhD+XfVca22+XTX6tsMN4als0GI3gLEi8wSLtrahj9+kfg"
    "jAUIWMCA5awwgFQCJc4QcYako53zEML8UXiiZERg+jBQRn7Q4Dwi9LAJRApAhNUfxb6IAr5bgfFTzeqxS/mMBb5m4Z6J/CK4"
    "hDUwViCfxfkz6YGZwjPDZkENKVeHHZKuYgeEx6MG5CsPw9jhxMOx2AEZHYKCwAkoRpPVjijXecKIxPAW5hOpieNhf14IAmZc"
    "TpleYST7Z+PScuwSi1pZG7IWGsF8gAAYe0Ke+f9r7/TaSAGvLQ+p/D5e/njb/3xZf3o97i6UOFuEKP0cEPhyVKTAyLEFYV7+"
    "RHJc7BEEaagUaEDnvJBf8xI+K3UbruAZuJGkkO7Tl9fXhdQAIL0EgdaoSyK05nMTBK+5JIj3EYnaYmH5grAuX2T/EvI5+3fm"
    "s4gUvlIgKvy9F4QYy7wAF80XzM43CNecYsKBUxCtUUG45iCMS1yV6grm7jypSV2YC5T7lzizF+RJYDxxXJUVvBgu1GXhnsDE"
    "XcHMDY7LsqSMqnfh2r9E7Hk3AqmrilQaDYoL73ZaYlTrB3z0GPfwYyXxgOLUwgQnfQuqmkbLmcWBreQSS6drT2JkFug2oJxh"
    "jq2xGJxrse5rWPG41jKqUl+1BNDyAJdy8SeZxTwIdxOtRNCZfoGWqSlCnXkCyaxncSoRIKSIpYWnJ/mJFsDgF6QBbEinlAgO"
    "UVUK0mQ2kDJQcbcETHEzR2O5+Zuz2ZyKPHd5Afew8DdhXt6EfYOtfvNpb09slftjxtGjgffTOjd4Tr41LQPBMr4pQJWcP7dm"
    "R897MprI46jgsUJUMRvTKIJaN8u5ocqjUIUpVIocd1DL6Ri04Ok0yhc+4amKyoQZ7JqwiaC+ZSeQPUWL5lwjolfPaI+nCyrY"
    "ADfFkCKEDksxcJWKVVbKb9UqqyWzi//xYJp34cDSyRoOB8StkgdjkuduDtaJviSPB37rEORM1M/OOavraSRm038gnuZgax0g"
    "fNmdI2wBwePyiZtoYWwlhKSnUlYsQlsp+L/vT8fzQsvBLqhWT46BtKNsu1/igHbBxz4a92jYoSBzpVxTQGpptDBwvpc3lrJW"
    "YBre5ghNY9ta3nDBfXWqJ1II9sv4OKE/eMNX1/DEjRwTMPRH5m2c44YdHjhsoavDhx/aMxCNtKK41nDkR/N2vFdYc4jpbxS+"
    "FSjZNCe9rg66JvAGAU7ca5KdHaz1R9vjwZriyURth9qYp6+KZHICtkH3kY0gpDf1nqlGxzLyy+dfzw1D2bk7n/e7Q5ygli9b"
    "ZfK522Zpm8E8z9MoQbBQb941eJYsqkH35zrqTTU2FvXc4YA99nN2ksScay4UBrC6ByIctefzRSPuS1XtcgQmWVjdeY/BEIcl"
    "vAocc0CocSIadt+RUyq4AzqZaGq2W894wHMZxmcB9T6gNzC6EmOQz0QqD4q09kDRhibMgQkSdMAdYscNA9gPEuD2KiHp9jE4"
    "v1Uvvfut5GFgWx2SbitrjHsAzCehbu6o6fH5aJPodPyPWGbPx1e5XqW51QXR3TLETdX/QaSoERlAcCi862IwpRvAyQScKYBL"
    "G11GBcaNDjTwPpLAkMpuNBHM3HgWCZs29Cq1sQDqfUA6uvjru4OA+H2SY2nfDVi/T/AzXTxWQRQvn3qCgqQX9RZiPejn1+yV"
    "RNTzTrRa28yloUCq8CNa6oUxplfwnBbDMk0raj81DH0emiZvJ2e66SHGLvbMS7c2+X3c3mser1gjZom27rjiDguMB2yeIGR9"
    "IuRsG7uf4M4ye1To6xP6yTSEHbMu4AgECbPkG+E30yt+swF7mD89bFZCPrVLiNHSwhliI1zs4F5iY5vxeYh7MBi13G1b6dQT"
    "4hlSZpURgqb2GJzlg2FVSaaYMPPk2418CqzzZSkdnGrStrtl9Ef4sWAzzQa3P2oi4FSlP8lPPiPkZSq8eamNMoF3DsBzvd2T"
    "C2rUM6IgxyttfgmmfOJFAcsiV1tmsEun89shGSCHXUMddSA64x3+qnUrH1VUEUfVdnuP1OZbBNoU0n9RO5t8GznhNoWrs33f"
    "TnedDReQEBWwnJx6Kd+LOLT8zLJllmdFIzjHwAWkD0AuxzIiVgB9MUmcqmY+nA7BPQCZxvav3mVPnSbBftk/Mlxn/8ryupNT"
    "+hhOzGltRB+lUkIULsuNI/pg1XyyGAlUv7y9CvTXQLQBGILAkMJdEHiZqB/4vWCTYlfWEyTChR1XGpFO7wWqCR2toN4iEWAb"
    "nirvBWbpKKqZl/3IzLsplP+1VQwsZN/fjYU3kHULWww/AhurdCEoaCu2ipnTAUc3jfNxsJYN0iqTEFX32FnBfuaq3fJwjrTw"
    "Cn+X8ReZXK02o+RHybeTGcar7PfzzZo2yN7nAVCJvEJUIk+MUnZamSIUyMvPxXBIZuZ7JiOWd9K/B0Vk5rZKZ5dHKtteHsmc"
    "eLDUZca72pnU6fBYpbWLe1S2u74ZDuTz21QnirttGPGntizUYB4dZJiZayxEyZ64ycRQu8o+RjNPeAcSq+f4PeQCbe8ghoKu"
    "AzYmiVkdKudcb2cnh7z1PgIbDREZB0rlc3kKHP7e5bhMQSuw8QiiaGrAJLmiGqN6ytGm1PdWnKhdqFZrvjbbzVMtre83lhUx"
    "Qw6VD9+byqOgV2q6SOV0hzAX6tmwkvVYrpzKlj3h0tZr6QopgkzNl7ufFfBAFEU6ObVzYcojp86P/an0YUgbyqndASKzwq9z"
    "BdU6o+WsaZbM4V+0Q/0ezi3MbtcVyh3d1JIn7mZBzUoXbGJFhcqP6qZbjgFevi0J+FImw3R2V8i0sPFE4MiN/Z9JwURRQPrC"
    "tF+WepK0sJW18U+K4+Pr6wJPx+tgTBs9yabWk73Jb19DiW9XJjFecK2yr7f2uZRoYenak6JhJdWFrEJSSHtP80GTzuckiVCs"
    "mFWmLpmj0tQiV8gci1lS+hivaVXoymO8LpRuTlVmxEjCXhbpT8i9M034Ohkr7YTktpsBD2DGHsL2qn6aeNobEzyt+r5SE9Yx"
    "uZQKN3lF1dytJzAMy2mEKGh3VxgoCrirb1mkPn5LC9yDQk2TnDHuDG93XBamCQNeE32c2qWqkdkUbL3FNEuZUpM5yRHLKY1V"
    "wZtCIbgm3LLxQkCz4if3QE1etg3WYPa3MfFvVYElm/G2IUKgcUF/gmU/ICjd6A2ln/G40csxVGhp3UAjBxtA1hGYKNhghrTp"
    "+T+sfCB4xNEI5cMHaw8eUj4xjz9G+RBc69Y2qs1Npdr1gGmKbbsebFvsgElKcFwkaBfOUP7QNupmbgNVRuua/R+yAWkrC7Ru"
    "ZKRQGVRduPme15TQertqqJap9wGol4DaExrbbE7HL4ePi6/SrMo+HU/Z1+zlIFOxt/BNHMjvnTj0tjcQXQBL2iHHxsfR3UF2"
    "wyRuJdvII5+BvK089xSAC8N5psSymJktKnlceGn6E9T0euWetoHREq6oup2mYcFB6fl/tvpmyGgI3YycKTiRpBDD3dozdGPb"
    "BVUWa8JYXZuuZ8N3Rg2VXEopRrpsAVkFvJ0VP8xpzN6uwsCLGiKppr0iIqa7jY4U7QuV8NHKzVPZH1dS9gYjvgBAl6rlXYXF"
    "bIp/ZYG5OJacq89VuKqQ+F+u0ApzJP7xkhDEeC3rotVNvEK0opCp2TwDu5NcP8gpE88WuIRsgc45ggPnWOBc2T6n+zF07qRU"
    "nRO8agwSWnRPaSyqghQlqupK0IWR9jnI++Gdh1nVfZjjwDkWOFe2XieYMHYdqZ6ZVAsbOQFVq5Tk3quHVKoAz3QBvfbd2hkV"
    "e5CjQT1Hg4d1+70bZXZKcsg6uY14TqwZh0JbRyp+467PbWAiN3KS9V+Gg04CbT77rPHkpq9RPVu7nmHU44/wDGUDVNeR1eQJ"
    "GNeQgIWgHEjbVdXsbjCXMGDvspFMbt5lfU5nbSirrZxcjkC9SkBudH9MIabnjUtNZWlVhUWz5qTbC8tLIXk5XAqW3PHG/VqF"
    "6kHMdbl5VO19exmHSAqGeJ1OVNWhUL8kCi6v606TZgbG3hvS5x/r3//wqvZL/sMwb2RXxt2VIpxAX7FJWRe5X1dBrqNlMuvq"
    "JpEhqqpU8DF0Nb8Ew1IYmNxiYDKdVHAtgDxOdiccSxcT70wACiHDRteTJCLkeFr1O00siO7CVadCJLtMpTWl7hSaQ0PfdDCw"
    "WkdY0QgrOmGa2oAaCka7vQqkqEqEMVdsl0PBh926mJ7zQmBDpCLYWAFCGHf6tR5ZKX9j+0LPG1JteogFJ4mvtbYPzRqs926V"
    "vxr1YejbVkXtg0QKlF3cqy7W9cPwLVeZ7M2ciB3WeT68Z3OVhBo5UEj3zadIlElmcpvZqYGBE9bH9dyC0Tmwn3cXqNDoWler"
    "gGXVW8RxEjrCq+E6CXPC4iyH6C/e0sNJ70oNYkr4pAk1dNodFrYJYcdkuAfsXpcqtDVVaCQNcvtTfivvd/2Eyva/FzIv/g6I"
    "hJ2hAur2KlP/RxOQMg5GcypWvkN391nCuo+ZVxqqYnP1I224uX7hrmMBJopSL3s8KEvLNeEPwhBZcakyuDQoKWjWgZzv+jFo"
    "yU4D9SjDxpgwHW7UYfTH4IWM2UKNCYMVLLrDEa1+iNycjU+Kt93p4qgBu7byFGzjbgfQZtMDpQned6CyzYuJ3gu391AbKC9H"
    "tsGyjYS5Bxg3YfYiTESsu3G75mwuOYFD7iSfj6Y3EbDdFW8ArGqI/ZeZH8hpPdh8v00fsC/aJmdxqjfoHx8I7BTopnD8xk7P"
    "9GzbgAx6MzUhsRXp9i5DIXfPNj1nSm0uqb5MPSypL1O7oOYjTwBCV/zdgMjumGFvx8zcoXJj+QhSlT1sQ+akRulVKADCpcun"
    "tgEOXk9bFtQwgCQJ4DqdXPV9X2WH7zcdOy+5xS/ea52jgXMscC69oBMMe0MM/BBilBaNKhkNa9RqTNBDMGENJZyGSW1x4A/B"
    "oW7EFarp4woYlt+qOXmkFVPw/LkbwVFYzdH+aDPIYXcqU4FI2/BsRytoyD9uhAM8210FLBPh1XGVdLrluDmFOU+GRbzKstv0"
    "njgLOj48EO67srsyoHHr3TZYQsHnEXCiuJlv7LzQ4M4LbTf4hVg8HQei7hRmtmXc1gwh0+7LmLj/8+DdmgEYyMGQ3bNB3rZN"
    "XaGpi52bAuzd4W33sa0ArfQX14b/LqGJaCqmHr6h/L3bSs1AVSZDVcKyKiOB6E0mUeDkOBkeN8XEb/NqBUCOI0E1v3qCu1vc"
    "CliSDCtxUd3AIbDmX/8FtQp4uw=="
)
print("Delta Drills checker ready — 40 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-numpy-broadcasting-rules -->

## Broadcasting rules

`numpy.broadcasting-rules`


<!-- dd:dd-seg-numpy-broadcasting-rules-0 -->

### the right-alignment rule


Elementwise operations "require matching shapes" — except PyTorch will
**stretch** certain mismatched shapes to fit, following one mechanical rule
set called **broadcasting**. It is the engine under half of idiomatic
tensor code,
and it is fully predictable:

> **Align the two shapes from the RIGHT. For each axis pair, the sizes are
> compatible if they are equal, or if one of them is 1** (that axis gets
> conceptually copied to match). **A missing leading axis counts as 1.**

Work `(m, 1) + (1, n)` by hand: align → axis 0 is m vs 1 (stretch the 1 → m),
axis 1 is 1 vs n (stretch → n) — result shape `(m, n)`, where entry [i, j] =
`a[i, 0] + b[0, j]`. Every pairwise-combination table, distance matrix, and
outer product starts exactly like this.

The stretching is *virtual* — no copies are made; PyTorch just reuses the
single row/column while iterating. Two everyday cases you have already been
using: scalar-with-array (`z * 2`) and matrix-with-row (`z - row` where row
has shape (n,) — aligned right, it matches z's last axis).


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\na = t.arange(3).reshape(3, 1)      # column: [[0], [1], [2]]           (3, 1)\nb = t.arange(4).reshape(1, 4)      # row:    [[0, 1, 2, 3]]            (1, 4)\n\n# Align right:  (3, 1)\n#               (1, 4)\n# axis 1: 1 vs 4 -> stretch a across columns; axis 0: 3 vs 1 -> stretch b\n# down rows. Result (3, 4): the "addition table" of the two vectors.\ntable = a + b\n\n# The result SHAPE is decidable from the two input shapes alone, with nothing\n# allocated — which is what you want when the real tensors are large and you\n# only need to know whether they will fit together.\n\n# Incompatible shapes fail here too, and fail early.\nprint("(3,1) + (1,4) ->", tuple(table.shape))\nprint(table)\n\ntry:\n    t.broadcast_shapes((3, 4), (5, 4))\nexcept RuntimeError as err:\n    print("RuntimeError:", err)\nelse:\n    raise AssertionError("(3, 4) and (5, 4) should not broadcast")\n# Hidden checks\nassert table.shape == (3, 4)\nassert table.tolist() == [[0, 1, 2, 3],\n                          [1, 2, 3, 4],\n                          [2, 3, 4, 5]]\nassert t.broadcast_shapes(a.shape, b.shape) == (3, 4)\n', globals()), end='')




Why: writing the two shapes one above the other, right-aligned, and
resolving each column IS the method — do it on paper until it's automatic.
The result shape falls out before any code runs, which is exactly what
`t.broadcast_shapes` is for: it answers the shape question without touching a
single element, so reach for it instead of building a result you intend to
throw away.


<!-- dd:dd-q111 -->

### Problem 111 · faded — your turn

The broadcast sum of a column (m, 1) and a row (1, n).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [1, 2, 3],
        [2, 3, 4]])
```


In [ ]:
import torch as t

def solve(a, b):
    """(m, n) table where entry [i, j] = a[i, 0] + b[0, j]."""
    return _____ + _____


# Example run — the grader calls solve() with several pairs.
print(solve(t.arange(3).reshape(3, 1), t.arange(3).reshape(1, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(111)


In [ ]:
#@title 💡 Solution — Problem 111
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(a, b):
    return a + b


print(solve(t.arange(3).reshape(3, 1), t.arange(3).reshape(1, 3)))


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\na = t.arange(3).reshape(3, 1)      # column: [[0], [1], [2]]           (3, 1)\nb = t.arange(4).reshape(1, 4)      # row:    [[0, 1, 2, 3]]            (1, 4)\n\n# Align right:  (3, 1)\n#               (1, 4)\n# axis 1: 1 vs 4 -> stretch a across columns; axis 0: 3 vs 1 -> stretch b\n# down rows. Result (3, 4): the "addition table" of the two vectors.\ntable = a + b\n\n# The result SHAPE is decidable from the two input shapes alone, with nothing\n# allocated — which is what you want when the real tensors are large and you\n# only need to know whether they will fit together.\n\n# Incompatible shapes fail here too, and fail early.\nprint("(3,1) + (1,4) ->", tuple(table.shape))\nprint(table)\n\ntry:\n    t.broadcast_shapes((3, 4), (5, 4))\nexcept RuntimeError as err:\n    print("RuntimeError:", err)\nelse:\n    raise AssertionError("(3, 4) and (5, 4) should not broadcast")\n# Hidden checks\nassert table.shape == (3, 4)\nassert table.tolist() == [[0, 1, 2, 3],\n                          [1, 2, 3, 4],\n                          [2, 3, 4, 5]]\nassert t.broadcast_shapes(a.shape, b.shape) == (3, 4)\n', globals()), end='')




Why: writing the two shapes one above the other, right-aligned, and
resolving each column IS the method — do it on paper until it's automatic.
The result shape falls out before any code runs, which is exactly what
`t.broadcast_shapes` is for: it answers the shape question without touching a
single element, so reach for it instead of building a result you intend to
throw away.


<!-- dd:dd-q949 -->

### Problem 949 · faded — your turn

The shape question, asked of the shapes themselves — there is no tensor
here to measure, so the answer has to come from the rule.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(3, 4)
```


In [ ]:
import torch as t


def solve(s1, s2):
    """The shape broadcasting s1 and s2 together produces."""
    return tuple(t._____(s1, s2))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve((3, 1), (1, 4)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(949)


In [ ]:
#@title 💡 Solution — Problem 949
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(s1, s2):
    """The shape broadcasting s1 and s2 together produces."""
    return tuple(t.broadcast_shapes(s1, s2))


print(solve((3, 1), (1, 4)))


<!-- dd:dd-seg-numpy-broadcasting-rules-1 -->

### placing the 1s yourself with None


The craft skill is **placing the 1s yourself**. Indexing with `None` (alias
`None`, which is what NumPy spells `np.newaxis`) inserts a length-1 axis:
`v[:, None]` turns shape (n,) into a
**column** (n, 1); `v[None, :]` makes an explicit **row** (1, n). When an
operation needs a vector to run *down* rather than *across* (or to hit a
specific axis of a 3-D array), you reshape it with `None` until the alignment
says what you mean.

When shapes are incompatible (say (3,) with (4,)), PyTorch raises rather than
guessing — a broadcast error means your alignment is wrong, and the fix is
almost always a well-placed `None`. Never reshape at random until the error
goes away: work the right-alignment on paper, decide where the 1 belongs,
and place it deliberately.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\n# The addition table again, from FLAT vectors — we place the 1-axes:\nva, vb = t.arange(3), t.arange(4)\ntable = va[:, None] + vb[None, :]\n\n# 3-D case: image (h, w, c) scaled per-PIXEL by map (h, w).\n# Align right: (2, 2, 3) vs (2, 2) -> trailing axes are 3 vs 2: INCOMPATIBLE.\n# The map needs its stretch-axis at the END: scale[:, :, None] is (2, 2, 1).\nimg = t.ones((2, 2, 3))\nscale = t.tensor([[1.0, 2.0],\n                  [3.0, 4.0]])\nscaled = img * scale[:, :, None]\nprint("flat vectors, 1-axes placed by hand ->", tuple(table.shape))\nprint(table)\nprint("scale", tuple(scale.shape), "-> scale[:, :, None]",\n      tuple(scale[:, :, None].shape), "-> img * it", tuple(scaled.shape))\nprint("pixel [1, 0] scaled by 3:", scaled[1, 0])\n# Hidden checks\nassert table.shape == (3, 4)\nassert scaled.shape == (2, 2, 3)\nassert scaled[1, 0].tolist() == [3.0, 3.0, 3.0]   # whole pixel scaled by 3\n', globals()), end='')




Why: the `va[:, None] + vb[None, :]` form is the general recipe for "all
pairs f(a_i, b_j)". In the 3-D case, the naive `img * scale` FAILS the
alignment check — working the rule shows the 1 must go at the end.


<!-- dd:dd-q151 -->

### Problem 151 · faded — your turn

Scale each pixel of an (h, w, c) image by a per-pixel (h, w) map.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[1., 1., 1.],
         [2., 2., 2.]],

        [[3., 3., 3.],
         [4., 4., 4.]]])
```


In [ ]:
import torch as t

def solve(a, b):
    """(h, w, c) image a scaled per-pixel by (h, w) map b."""
    return a * b[_____]


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 2, 3)), t.tensor([[1.0, 2.0], [3.0, 4.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(151)


In [ ]:
#@title 💡 Solution — Problem 151
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return a * b[:, :, None]


print(solve(t.ones((2, 2, 3)), t.tensor([[1.0, 2.0], [3.0, 4.0]])))


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\n# The addition table again, from FLAT vectors — we place the 1-axes:\nva, vb = t.arange(3), t.arange(4)\ntable = va[:, None] + vb[None, :]\n\n# 3-D case: image (h, w, c) scaled per-PIXEL by map (h, w).\n# Align right: (2, 2, 3) vs (2, 2) -> trailing axes are 3 vs 2: INCOMPATIBLE.\n# The map needs its stretch-axis at the END: scale[:, :, None] is (2, 2, 1).\nimg = t.ones((2, 2, 3))\nscale = t.tensor([[1.0, 2.0],\n                  [3.0, 4.0]])\nscaled = img * scale[:, :, None]\nprint("flat vectors, 1-axes placed by hand ->", tuple(table.shape))\nprint(table)\nprint("scale", tuple(scale.shape), "-> scale[:, :, None]",\n      tuple(scale[:, :, None].shape), "-> img * it", tuple(scaled.shape))\nprint("pixel [1, 0] scaled by 3:", scaled[1, 0])\n# Hidden checks\nassert table.shape == (3, 4)\nassert scaled.shape == (2, 2, 3)\nassert scaled[1, 0].tolist() == [3.0, 3.0, 3.0]   # whole pixel scaled by 3\n', globals()), end='')




Why: the `va[:, None] + vb[None, :]` form is the general recipe for "all
pairs f(a_i, b_j)". In the 3-D case, the naive `img * scale` FAILS the
alignment check — working the rule shows the 1 must go at the end.


<!-- dd:dd-q948 -->

### Problem 948 · faded — your turn

The pairwise table again, and this time the operation is a difference —
which means the order of the two operands matters.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ -9., -19.],
        [ -8., -18.],
        [ -7., -17.]])
```


In [ ]:
import torch as t


def solve(a, b):
    """(m, n) table where entry [i, j] = a[i] - b[j]."""
    return a[_____] - b[_____]


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([1.0, 2.0, 3.0]), t.tensor([10.0, 20.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(948)


In [ ]:
#@title 💡 Solution — Problem 948
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(a, b):
    """(m, n) table where entry [i, j] = a[i] - b[j]."""
    return a[:, None] - b[None, :]


print(solve(t.tensor([1.0, 2.0, 3.0]), t.tensor([10.0, 20.0])))


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\n# The addition table again, from FLAT vectors — we place the 1-axes:\nva, vb = t.arange(3), t.arange(4)\ntable = va[:, None] + vb[None, :]\n\n# 3-D case: image (h, w, c) scaled per-PIXEL by map (h, w).\n# Align right: (2, 2, 3) vs (2, 2) -> trailing axes are 3 vs 2: INCOMPATIBLE.\n# The map needs its stretch-axis at the END: scale[:, :, None] is (2, 2, 1).\nimg = t.ones((2, 2, 3))\nscale = t.tensor([[1.0, 2.0],\n                  [3.0, 4.0]])\nscaled = img * scale[:, :, None]\nprint("flat vectors, 1-axes placed by hand ->", tuple(table.shape))\nprint(table)\nprint("scale", tuple(scale.shape), "-> scale[:, :, None]",\n      tuple(scale[:, :, None].shape), "-> img * it", tuple(scaled.shape))\nprint("pixel [1, 0] scaled by 3:", scaled[1, 0])\n# Hidden checks\nassert table.shape == (3, 4)\nassert scaled.shape == (2, 2, 3)\nassert scaled[1, 0].tolist() == [3.0, 3.0, 3.0]   # whole pixel scaled by 3\n', globals()), end='')




Why: the `va[:, None] + vb[None, :]` form is the general recipe for "all
pairs f(a_i, b_j)". In the 3-D case, the naive `img * scale` FAILS the
alignment check — working the rule shows the 1 must go at the end.


<!-- dd:dd-q950 -->

### Problem 950 · faded — your turn

A step past the image case: the stretch axis is the FIRST one, so the 1s
go after it — and there are two of them to add, not one.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[ 1.,  1.,  1.],
         [ 1.,  1.,  1.]],

        [[10., 10., 10.],
         [10., 10., 10.]]])
```


In [ ]:
import torch as t


def solve(x, s):
    """Every element of sample i scaled by s[i]."""
    return x * s[_____]


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.ones(2, 2, 3), t.tensor([1.0, 10.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(950)


In [ ]:
#@title 💡 Solution — Problem 950
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(x, s):
    """Every element of sample i scaled by s[i]."""
    return x * s[:, None, None]


print(solve(t.ones(2, 2, 3), t.tensor([1.0, 10.0])))


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\n# The addition table again, from FLAT vectors — we place the 1-axes:\nva, vb = t.arange(3), t.arange(4)\ntable = va[:, None] + vb[None, :]\n\n# 3-D case: image (h, w, c) scaled per-PIXEL by map (h, w).\n# Align right: (2, 2, 3) vs (2, 2) -> trailing axes are 3 vs 2: INCOMPATIBLE.\n# The map needs its stretch-axis at the END: scale[:, :, None] is (2, 2, 1).\nimg = t.ones((2, 2, 3))\nscale = t.tensor([[1.0, 2.0],\n                  [3.0, 4.0]])\nscaled = img * scale[:, :, None]\nprint("flat vectors, 1-axes placed by hand ->", tuple(table.shape))\nprint(table)\nprint("scale", tuple(scale.shape), "-> scale[:, :, None]",\n      tuple(scale[:, :, None].shape), "-> img * it", tuple(scaled.shape))\nprint("pixel [1, 0] scaled by 3:", scaled[1, 0])\n# Hidden checks\nassert table.shape == (3, 4)\nassert scaled.shape == (2, 2, 3)\nassert scaled[1, 0].tolist() == [3.0, 3.0, 3.0]   # whole pixel scaled by 3\n', globals()), end='')




Why: the `va[:, None] + vb[None, :]` form is the general recipe for "all
pairs f(a_i, b_j)". In the 3-D case, the naive `img * scale` FAILS the
alignment check — working the rule shows the 1 must go at the end.


<!-- dd:dd-q951 -->

### Problem 951 · faded — your turn

The row range has to run DOWN and the column range ACROSS. Both are flat
to begin with, so both need a 1 placed by hand.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
```


In [ ]:
import torch as t


def solve(rows, cols):
    """(rows, cols) grid where entry [i, j] = i * cols + j."""
    return t.arange(rows)[_____] * cols + t.arange(cols)[_____]


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(3, 4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(951)


In [ ]:
#@title 💡 Solution — Problem 951
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(rows, cols):
    """(rows, cols) grid where entry [i, j] = i * cols + j."""
    return t.arange(rows)[:, None] * cols + t.arange(cols)[None, :]


print(solve(3, 4))


<!-- dd:dd-q499 -->

### Problem 499 · guided

Write a function solve(x, bias) where x has shape (rows, cols) and bias has shape (cols,), returning x with bias added to EVERY row. No loop and no tiling: shapes line up from the RIGHT, so a length-cols vector already matches x's last axis.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[11., 22.],
        [13., 24.]])
```


<details>
<summary>Hints</summary>

1. Do NOT tile the bias. Write the addition and let the shapes meet.
2. Shapes align from the RIGHT: a (cols,) vector already lines up with the
   last axis of a (rows, cols) matrix, and the missing axis is treated as 1.
3. `x + bias`.

</details>


In [ ]:
import torch as t

def solve(x, bias):
    """Add a per-column bias vector to every row."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 20.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(499)


In [ ]:
#@title 💡 Solution — Problem 499
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, bias):
    """Add a per-column bias vector to every row."""
    return x + bias


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 20.0]))
print(solve(*example))


<!-- dd:dd-q500 -->

### Problem 500 · guided

Write a function solve(a, b) that returns, as a plain tuple of ints, the shape you get from combining a and b elementwise. Answer it without building the result: the shape follows from the two input shapes alone, and materialising a combination you are going to throw away can cost real memory on real tensors.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(3, 4)
```


<details>
<summary>Hints</summary>

1. You are asked for the resulting shape, not for a rule recited back.
2. There is a function that takes the two SHAPES and returns the combined one,
   without building anything — then convert its torch.Size to a plain tuple.
3. `tuple(t.broadcast_shapes(a.shape, b.shape))`.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return the shape broadcasting a and b together produces."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.zeros(3, 1), t.zeros(1, 4))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(500)


In [ ]:
#@title 💡 Solution — Problem 500
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return the shape broadcasting a and b together produces."""
    return tuple(t.broadcast_shapes(a.shape, b.shape))



example = (t.zeros(3, 1), t.zeros(1, 4))
print(solve(*example))


<!-- dd:dd-q952 -->

### Problem 952 · guided

Two 1-D PyTorch tensors are given: `a` of length m and `b` of length n. Return the (m, n) tensor whose entry at row i, column j is the i-th entry of a times the j-th entry of b — the outer product of the two vectors. No loops, and do not call a matrix-multiply or outer-product helper.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[10., 20.],
        [20., 40.],
        [30., 60.]])
```


<details>
<summary>Hints</summary>

1. Both inputs are FLAT, so neither of them runs down the page yet. Aligned right as they are, (m,) against (n,), the rule pairs m with n and refuses.
2. One of them has to become a column and the other a row. `v[:, None]` is the column form, `v[None, :]` the row form.
3. `a[:, None] * b[None, :]`.

</details>


In [ ]:
import torch as t


def solve(a, b):
    """(m, n) table where entry [i, j] = a[i] * b[j]."""
    return  # your answer here

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([1.0, 2.0, 3.0]), t.tensor([10.0, 20.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(952)


In [ ]:
#@title 💡 Solution — Problem 952
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(a, b):
    """(m, n) table where entry [i, j] = a[i] * b[j]."""
    return a[:, None] * b[None, :]


print(solve(t.tensor([1.0, 2.0, 3.0]), t.tensor([10.0, 20.0])))


<!-- dd:dd-q953 -->

### Problem 953 · guided

`imgs` has shape (b, h, w, c) — a batch of images with the colour axis LAST — and `mean` has shape (c,), one value per colour channel. Return `imgs` with each channel's mean subtracted from that channel. One expression, no loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[ 1.,  0., -1.],
          [ 1.,  0., -1.]],

         [[ 1.,  0., -1.],
          [ 1.,  0., -1.]]]])
```


<details>
<summary>Hints</summary>

1. Write the two shapes one above the other and align them from the RIGHT: (b, h, w, c) over (c,).
2. Every column of that alignment already resolves — c pairs with c, and the three missing leading axes each count as 1. Nothing needs placing by hand.
3. `imgs - mean`.

</details>


In [ ]:
import torch as t


def solve(imgs, mean):
    """Each channel of every image centred by that channel's mean."""
    return  # your answer here

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.ones(1, 2, 2, 3), t.tensor([0.0, 1.0, 2.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(953)


In [ ]:
#@title 💡 Solution — Problem 953
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(imgs, mean):
    """Each channel of every image centred by that channel's mean."""
    return imgs - mean


print(solve(t.ones(1, 2, 2, 3), t.tensor([0.0, 1.0, 2.0])))


<!-- dd:dd-q501 -->

### Problem 501 · independent

Write a function solve(x, scale) where x has shape (rows, cols) and scale has shape (rows,), returning x with row i multiplied by scale[i]. Right-alignment would pair scale against the COLUMNS, which is not what you want — insert a length-1 axis with None so it lines up against the rows instead.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 10.,  20.],
        [300., 400.]])
```


In [ ]:
import torch as t

def solve(x, scale):
    """Scale every ROW by its own factor."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 100.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(501)


In [ ]:
#@title 💡 Solution — Problem 501
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, scale):
    """Scale every ROW by its own factor."""
    return x * scale[:, None]


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 100.0]))
print(solve(*example))


<!-- dd:dd-q502 -->

### Problem 502 · independent

Write a function solve(rows, cols) returning a (rows, cols) integer tensor whose [i][j] entry is i*j. Build two 1-D ranges and give each a length-1 axis on the side the other one varies along — the outer product falls out of broadcasting, with no loop and no tiling.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 0, 0, 0],
        [0, 1, 2, 3],
        [0, 2, 4, 6]])
```


In [ ]:
import torch as t

def solve(rows, cols):
    """Build a multiplication table from two 1-D ranges."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 4)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(502)


In [ ]:
#@title 💡 Solution — Problem 502
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols):
    """Build a multiplication table from two 1-D ranges."""
    r = t.arange(rows)
    c = t.arange(cols)
    return r[:, None] * c[None, :]


example = (3, 4)
print(solve(*example))


<!-- dd:dd-q954 -->

### Problem 954 · independent

`x` has shape (rows, cols), `row_scale` has shape (rows,) and `col_bias` has shape (cols,). Return `x` with row i multiplied by `row_scale[i]` and then `col_bias[j]` added to column j. One expression, no loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 1.,  2.,  3.],
        [10., 11., 12.]])
```


In [ ]:
import torch as t


def solve(x, row_scale, col_bias):
    """Rows scaled by row_scale, then col_bias added along the columns."""
    return  # your answer here

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.ones(2, 3), t.tensor([1.0, 10.0]), t.tensor([0.0, 1.0, 2.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(954)


In [ ]:
#@title 💡 Solution — Problem 954
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(x, row_scale, col_bias):
    """Rows scaled by row_scale, then col_bias added along the columns."""
    return x * row_scale[:, None] + col_bias


print(solve(t.ones(2, 3), t.tensor([1.0, 10.0]), t.tensor([0.0, 1.0, 2.0])))


<!-- dd:dd-q955 -->

### Problem 955 · independent

Three 1-D PyTorch tensors are given: `a` of length i, `b` of length j and `c` of length k. Return the (i, j, k) tensor whose entry at position p, q, r is the product of the p-th entry of a, the q-th entry of b and the r-th entry of c. One expression, no loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[1000.],
         [2000.],
         [3000.]],

        [[2000.],
         [4000.],
         [6000.]]])
```


In [ ]:
import torch as t


def solve(a, b, c):
    """(i, j, k) tensor where entry [p, q, r] = a[p] * b[q] * c[r]."""
    return  # your answer here

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([1.0, 2.0]), t.tensor([10.0, 20.0, 30.0]), t.tensor([100.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(955)


In [ ]:
#@title 💡 Solution — Problem 955
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(a, b, c):
    """(i, j, k) tensor where entry [p, q, r] = a[p] * b[q] * c[r]."""
    return a[:, None, None] * b[None, :, None] * c[None, None, :]


print(solve(t.tensor([1.0, 2.0]), t.tensor([10.0, 20.0, 30.0]), t.tensor([100.0])))


#### Common mistakes

- **"Broadcasting matches shapes from the left."** — From the RIGHT. `(3,)`
  against `(3, 4)` aligns 3-with-4 and fails; `(4,)` against `(3, 4)` aligns
  4-with-4 and works. Most surprise errors are left-alignment intuition.
- **"Stretching copies the data."** — The stretch is virtual; memory is
  reused, not duplicated. Broadcasting a (10000, 1) against (1, 10000) does
  NOT allocate 10⁸ intermediate elements for the inputs.
- **"When shapes don't broadcast, reshape until the error goes away."** —
  Random reshaping produces silently WRONG results more often than errors.
  Work the right-alignment on paper, decide where the 1 belongs, and place it
  with `None` deliberately.


<!-- dd:dd-kp-numpy-axis-reductions -->

## Reductions along an axis — and keepdims

`numpy.axis-reductions`


<!-- dd:dd-seg-numpy-axis-reductions-0 -->

### axis= — the axis you name disappears


Whole-array reductions collapse everything to one number. Add **`axis=`** and
the reduction collapses **only that axis**, leaving the rest of the shape
intact:

> **The axis you name is the axis that DISAPPEARS.**

For a (r, c) matrix:

- `x.sum(axis=0)` — axis 0 (rows) disappears → shape (c,): **column sums**
  (you summed *down* each column).
- `x.sum(axis=1)` — axis 1 disappears → shape (r,): **row sums**.

The naming feels backwards until you anchor it: `axis=0` does NOT mean
"per-row results", it means "reduce ALONG axis 0" — the r rows are collapsed
on top of each other. Predict the output shape first (cross the named axis
out of the shape tuple) and the direction sorts itself out.

Everything from the aggregation KP takes `dim=`: `mean`, `amin`, `amax`,
`std`, `any`, `all`, `argmax`, plus `t.quantile` and friends. Two PyTorch
wrinkles carry through this whole KP: `mean` refuses an integer tensor (cast
with `.to(t.float32)` first), and `std` divides by n−1 unless you pass
`correction=0`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nx = t.tensor([[1.0, 2.0, 3.0],\n              [10.0, 20.0, 30.0]])    # shape (2, 3) — float, so mean works\n\n# dim=0 -> the 2 rows collapse onto each other -> one sum PER COLUMN.\ncol_sums = x.sum(dim=0)\n\n# dim=1 -> the 3 columns collapse -> one value PER ROW.\nrow_means = x.mean(dim=1)\nprint("x shape", tuple(x.shape))\nprint("sum(dim=0) ", col_sums,  "shape", tuple(col_sums.shape))\nprint("mean(dim=1)", row_means, "shape", tuple(row_means.shape))\n# Hidden checks\nassert tuple(col_sums.shape) == (3,)  # (2, 3) with dim 0 crossed out\nassert col_sums.tolist() == [11.0, 22.0, 33.0]\nassert tuple(row_means.shape) == (2,)\nassert row_means.tolist() == [2.0, 20.0]\n', globals()), end='')




Why: for each reduction, the assert on `.shape` comes BEFORE the values —
that's the recommended order in your own code too: predict the shape by
crossing out the named axis, then check the numbers.


<!-- dd:dd-q220 -->

### Problem 220 · faded — your turn

One sum per column.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([11, 22, 33])
```


In [ ]:
import torch as t

def solve(x):
    """Column sums of a 2-D matrix: which axis disappears?"""
    return x.sum(_____=_____)


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(220)


In [ ]:
#@title 💡 Solution — Problem 220
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.sum(dim=0)


example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


<!-- dd:dd-seg-numpy-axis-reductions-1 -->

### tuples of axes, and keepdims


Higher-rank arrays allow a *tuple* of axes — `x.sum(axis=(-2, -1))` collapses
the last two dimensions at once (e.g. summing each image of a batch), and
negative indices count from the end just like in indexing. That makes
"per-image" reductions one call, robust to how many leading batch axes exist.

One more switch on the same call: **`keepdims=True`** keeps the reduced axis
as length 1 instead of deleting it — shape (r, c) → (r, 1) rather than (r,).
Why you'd want that: a (r, 1) result broadcasts back against the original
(r, c) *by row*. The reduce → keepdims → operate pipeline is the heart of the
next KP (centering), where you'll practice it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\n# Tuple of axes on a 4-D batch (a, b, c, d): collapse the last two ->\n# one total per (a, b) slice. Negative axes save counting.\nbatch = t.arange(24).reshape(2, 3, 2, 2)\ntotals = batch.sum(dim=(-2, -1))\n\n# keepdim preview: the reduced dim survives as 1, so the result still\n# lines up against the original for broadcasting.\n# (t.mean needs a float tensor — it will not promote ints the way numpy does.)\nx = t.tensor([[1.0, 2.0, 3.0], [10.0, 20.0, 30.0]])\nrm = x.mean(dim=1, keepdim=True)\ncentered = x - rm                     # (2,3) - (2,1): broadcasts by row\nprint("(2,3,2,2) summed over the last two ->", tuple(totals.shape))\nprint(totals)\nprint("keepdim=True keeps the axis as 1:", tuple(rm.shape), "->", rm.tolist())\nprint(centered)\n# Hidden checks\nassert totals.shape == (2, 3)\nassert totals[0, 0] == 0 + 1 + 2 + 3\nassert tuple(rm.shape) == (2, 1)\nassert centered[0].tolist() == [-1.0, 0.0, 1.0]\n', globals()), end='')




Why: a bare `(2,)` row-mean would align against the WRONG axis when
broadcast (right-aligned → columns) — the source of a classic silent bug
when r = c. keepdims makes the intended alignment explicit.


<!-- dd:dd-q135 -->

### Problem 135 · faded — your turn

Per-slice totals of a 4-D batch: collapse the LAST two axes in one call.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 6, 22, 38],
        [54, 70, 86]])
```


In [ ]:
import torch as t

def solve(x):
    """(a, b, c, d) -> (a, b): total of each c*d slice."""
    return x.sum(_____=_____)


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.arange(24).reshape(2, 3, 2, 2)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(135)


In [ ]:
#@title 💡 Solution — Problem 135
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.sum(dim=(-2, -1))


example = t.arange(24).reshape(2, 3, 2, 2)
print(solve(example))


<!-- dd:dd-q503 -->

### Problem 503 · guided

Write a function solve(x) that takes a 2-D integer tensor and returns a 1-D tensor holding the sum of each row, so the result has one entry per row. The axis you name is the one that DISAPPEARS — name the columns to collapse them.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 6, 60])
```


<details>
<summary>Hints</summary>

1. One number per row means the COLUMN axis has to go.
2. The axis you name in `dim=` is the one that disappears, so name the one
   you are collapsing, not the one you are keeping.
3. `x.sum(dim=1)`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the sum of each ROW."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(503)


In [ ]:
#@title 💡 Solution — Problem 503
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return the sum of each ROW."""
    return x.sum(dim=1)


example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


<!-- dd:dd-q504 -->

### Problem 504 · guided

Write a function solve(x) that returns the per-row sums of a 2-D tensor but with the reduced axis kept as a length-1 axis, so a (3, 4) input gives a (3, 1) result rather than (3,). Keeping it is what lets the answer broadcast back against the original — which is the whole reason keepdim exists.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 6],
        [60]])
```


<details>
<summary>Hints</summary>

1. Same reduction as before, but the result must still have two axes.
2. There is a keyword that leaves the collapsed axis behind at length 1,
   which is exactly what a later broadcast needs.
3. `x.sum(dim=1, keepdim=True)`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the row sums with the reduced axis KEPT."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(504)


In [ ]:
#@title 💡 Solution — Problem 504
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return the row sums with the reduced axis KEPT."""
    return x.sum(dim=1, keepdim=True)


example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


<!-- dd:dd-q108 -->

### Problem 108 · independent

Write a function solve(x, qs) that takes a 2-D PyTorch float tensor x of shape (n_rows, n_cols) and a list of quantile fractions qs (each in [0, 1]). Return a 2-D tensor of shape (len(qs), n_cols) where row i holds the qs[i] quantile of each column of x, using PyTorch's default (linear) interpolation.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 1.5000, 15.0000],
        [ 2.0000, 20.0000],
        [ 2.5000, 25.0000]])
```


In [ ]:
import torch as t

def solve(x, qs):
    """Return per-column quantiles of x, one row per requested fraction."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
print(solve(example, [0.25, 0.5, 0.75]))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(108)


In [ ]:
#@title 💡 Solution — Problem 108
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(x, qs):
    return t.quantile(x, t.tensor(qs, dtype=x.dtype), dim=0)


example = t.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
print(solve(example, [0.25, 0.5, 0.75]))


<!-- dd:dd-q505 -->

### Problem 505 · independent

Write a function solve(x) that takes a 2-D float tensor with no zero row-sums and returns a tensor of the same shape in which every row sums to 1. Reduce along the columns, keep the axis so the result still has two, and divide — the length-1 axis broadcasts back across the row it came from.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0.2500, 0.7500],
        [0.5000, 0.5000]])
```


In [ ]:
import torch as t

def solve(x):
    """Make every row of a float tensor sum to 1."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 3.0], [2.0, 2.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(505)


In [ ]:
#@title 💡 Solution — Problem 505
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Make every row of a float tensor sum to 1."""
    return x / x.sum(dim=1, keepdim=True)


example = t.tensor([[1.0, 3.0], [2.0, 2.0]])
print(solve(example))


<!-- dd:dd-q174 -->

### Problem 174 · independent

Write a function solve(z) that takes a 2-D float tensor with an ODD number of columns and returns a 1-D tensor holding each row's MEDIAN — the middle value of that row when sorted.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2.])
```


In [ ]:
import torch as t

def solve(z):
    """Return the median of each row of z (odd column count)."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[3.0, 1.0, 2.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(174)


In [ ]:
#@title 💡 Solution — Problem 174
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z):
    return t.median(z, dim=1).values


print(solve(t.tensor([[3.0, 1.0, 2.0]])))


#### Common mistakes

- **"axis=0 gives row sums."** — axis=0 REMOVES axis 0: the rows collapse
  together, yielding one result per column. Cross the axis out of the shape
  tuple and read what's left.
- **"keepdims is cosmetic."** — It preserves alignment for broadcasting.
  `x - x.mean(dim=1)` on a square matrix runs WITHOUT error and quietly
  subtracts along the wrong axis; `keepdims=True` (shape (r,1)) makes the
  intended row-wise alignment explicit and correct.
- **"Reducing two axes needs two calls."** — `axis=(1, 2)` collapses both in
  one pass. Chained single-axis calls also shift the axis numbering between
  calls — a tuple avoids that trap entirely.


<!-- dd:dd-kp-numpy-dot-matmul-patterns -->

## Dot products and matrix-multiply patterns

`numpy.dot-matmul-patterns`


<!-- dd:dd-seg-numpy-dot-matmul-patterns-0 -->

### the dot product — multiply, then sum


The **dot product** of two equal-length vectors — multiply corresponding
entries, add them up — is the atom that all of linear algebra's products are
built from. PyTorch spells it three interchangeable ways:

```python no-run
t.dot(a, b)      ==  a @ b  ==  (a * b).sum()
```

The third spelling is the important one conceptually: *dot = elementwise
multiply + reduction.* Holding the decomposition lets you build variants
(weighted dots, masked dots, batch dots) instead of hunting for a function
that may not exist.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\na = t.tensor([1.0, 2.0, 3.0])\nb = t.tensor([4.0, -5.0, 6.0])\n\n# The atom, three spellings — same number.\nd = float(t.dot(a, b))\nprint("a * b (no sum yet):", a * b)\nprint("t.dot:", d, "| a @ b:", float(a @ b), "| (a*b).sum():",\n      float((a * b).sum()))\n# Hidden checks\nassert d == float(a @ b) == float((a * b).sum()) == 12.0\n', globals()), end='')




Why: verifying the three dot spellings agree once buys permanent fluency:
when you see `(x * w).sum()` in someone's code, you now read "dot".


<!-- dd:dd-q37 -->

### Problem 37 · faded — your turn

Dot product of two vectors, as a plain float.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
12.0
```


In [ ]:
import torch as t

def solve(a, b):
    """The dot product of vectors a and b."""
    return float(t._____(a, b))


# Example run — the grader calls solve() with several different vector pairs,
# including edge cases. Your function must work for all of them.
a_example = t.tensor([1.0, 2.0, 3.0])
b_example = t.tensor([4.0, -5.0, 6.0])
print(solve(a_example, b_example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(37)


In [ ]:
#@title 💡 Solution — Problem 37
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return float(t.dot(a, b))


a_example = t.tensor([1.0, 2.0, 3.0])
b_example = t.tensor([4.0, -5.0, 6.0])
print(solve(a_example, b_example))


<!-- dd:dd-seg-numpy-dot-matmul-patterns-1 -->

### matrix @ vector — one dot per row


**Matrix @ vector** (`z @ v`, shapes (n, m) @ (m,)): one dot per row of z —
"each row dotted with v" in a single call. Result shape (n,). (Matrix @
matrix is one dot per (row, column) pair — the previous KP.)

Check one output by hand and the shape rule follows: matmul = a dot per
row, so an (n, m) matrix against a length-m vector yields n dots.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\n# Matrix @ vector: row i of the result = (row i of z) . v\nz = t.tensor([[1.0, 2.0],\n              [3.0, 4.0]])\nv = t.tensor([10.0, 1.0])\nzv = z @ v\nprint(tuple(z.shape), "@", tuple(v.shape), "->", tuple(zv.shape), ":", zv)\n# Hidden checks\nassert zv.tolist() == [12.0, 34.0]        # 1*10+2*1, 3*10+4*1\n', globals()), end='')




Why: checking one output by hand (1·10 + 2·1 = 12) anchors "matmul = a dot
per row" — and predicts the result's shape (n,) without memorizing another
rule.


<!-- dd:dd-q144 -->

### Problem 144 · faded — your turn

Each row of z dotted with v.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([12., 34.])
```


In [ ]:
import torch as t

def solve(z, v):
    """Length-n array: entry i = (row i of z) . v."""
    return z _____ v


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 1.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(144)


In [ ]:
#@title 💡 Solution — Problem 144
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, v):
    return z @ v


print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 1.0])))


<!-- dd:dd-seg-numpy-dot-matmul-patterns-2 -->

### when @ doesn't fit — multiply, then reduce an axis


When the pattern you need is *not* one of the packaged shapes, the
decomposition rescues you. "Dot each row of `a` with the CORRESPONDING row
of `b`" (same shapes) is not `a @ b` — matmul dots every row with every
COLUMN. But per the atom: multiply elementwise, then reduce each row:

```python no-run
(a * b).sum(dim=1)          # row-wise dots
```

That multiply-then-reduce-an-axis maneuver covers the "batch of dots" tasks
— and a later course will name that whole move in one string, with the
summed axis simply left off the output.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\n# Row-wise dots of two SAME-SHAPE matrices: NOT a matmul.\np = t.tensor([[1.0, 2.0],\n              [3.0, 4.0]])\nq = t.tensor([[5.0, 6.0],\n              [7.0, 8.0]])\nrow_dots = (p * q).sum(dim=1)\nprint("row-wise dots", row_dots)\nprint("p @ q would be something else entirely:")\nprint(p @ q)\n# Hidden checks\nassert row_dots.tolist() == [17.0, 53.0]  # 1*5+2*6, 3*7+4*8\n', globals()), end='')




Why: this case is deliberately a trap — `p @ q` runs on these square
matrices and returns the WRONG thing (full matrix product). Decomposing to
multiply+sum(axis=1) is the general escape whenever the packaged products
don't match the pairing you need.


<!-- dd:dd-q121 -->

### Problem 121 · faded — your turn

Dot each row of a with the corresponding row of b.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([11.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Length-n array: entry i = (row i of a) . (row i of b)."""
    return (a * b).sum(dim=_____)


# Example run — the grader calls solve() with several pairs.
print(solve(t.tensor([[1.0, 2.0]]), t.tensor([[3.0, 4.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(121)


In [ ]:
#@title 💡 Solution — Problem 121
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('ij,ij->i', a, b)


print(solve(t.tensor([[1.0, 2.0]]), t.tensor([[3.0, 4.0]])))


<!-- dd:dd-q514 -->

### Problem 514 · guided

Write a function solve(a, b) that takes two 1-D float tensors of the same length and returns their dot product as a plain Python float — but build it from the two steps it actually is: multiply the pairs, then sum the results. Seeing the pattern in the open is what lets you recognise it later in shapes that @ will not take.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
12.0
```


<details>
<summary>Hints</summary>

1. Do not reach for the built-in dot product — the question wants the two
   steps it is made of.
2. Multiply the pairs elementwise, then collapse the result to one number.
3. `float((a * b).sum())`.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return the dot product built by hand: multiply, then sum."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.0, 2.0, 3.0]), t.tensor([4.0, -5.0, 6.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(514)


In [ ]:
#@title 💡 Solution — Problem 514
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return the dot product built by hand: multiply, then sum."""
    return float((a * b).sum())


example = (t.tensor([1.0, 2.0, 3.0]), t.tensor([4.0, -5.0, 6.0]))
print(solve(*example))


<!-- dd:dd-q141 -->

### Problem 141 · independent

Write a function solve(a, b) that takes two square float matrices of the same shape (n, n) and returns a 1-D tensor of length n holding the DIAGONAL of the matrix product a @ b — without computing the full product (entry i is the dot product of row i of a with column i of b). Any approach producing those values passes, but the einsum form is the one worth learning.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([19., 50.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return diag(a @ b) as a 1-D array."""
    return None


# Example run — the grader calls solve() with several pairs.
example_a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
example_b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(example_a, example_b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(141)


In [ ]:
#@title 💡 Solution — Problem 141
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('ij,ji->i', a, b)


example_a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
example_b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(example_a, example_b))


<!-- dd:dd-q515 -->

### Problem 515 · independent

Write a function solve(x) that takes a 2-D float tensor with no all-zero rows and returns it with each row rescaled to length 1. A row's length is the square root of its dot product with itself — reduce along the columns, keep the axis so it broadcasts back, and divide.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0.6000, 0.8000],
        [0.0000, 1.0000]])
```


In [ ]:
import torch as t

def solve(x):
    """Scale every row to unit length."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[3.0, 4.0], [0.0, 2.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(515)


In [ ]:
#@title 💡 Solution — Problem 515
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Scale every row to unit length."""
    return x / x.pow(2).sum(dim=1, keepdim=True).sqrt()


example = t.tensor([[3.0, 4.0], [0.0, 2.0]])
print(solve(example))


<!-- dd:dd-q95 -->

### Problem 95 · independent

Write a function solve(img) that takes an RGB image as a 3-D PyTorch tensor of shape (h, w, 3) and returns the 2-D grayscale version of shape (h, w), where each output pixel is the weighted sum 0.299*R + 0.587*G + 0.114*B of that pixel's three channels. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[255.,   0.],
        [  0.,   0.]])
```


In [ ]:
import torch as t

def solve(img):
    """Return the (h, w) grayscale of an (h, w, 3) RGB image."""
    return None


# Example run — the grader calls solve() with several images.
example = t.zeros((2, 2, 3))
example[0, 0] = t.tensor([255, 255, 255])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(95)


In [ ]:
#@title 💡 Solution — Problem 95
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(img):
    return img @ t.tensor([0.299, 0.587, 0.114])


example = t.zeros((2, 2, 3))
example[0, 0] = t.tensor([255, 255, 255])
print(solve(example))


#### Common mistakes

- **"Row-wise dots of two matrices = a @ b."** — Matmul dots every row with
  every COLUMN. Corresponding-rows pairing is elementwise-multiply + row
  reduction: `(a * b).sum(axis=1)`.
- **"The dot product is its own primitive."** — It's multiply + sum. Holding
  the decomposition lets you build variants (weighted dots, masked dots,
  batch dots) instead of hunting for a function that may not exist.
- **"norm of a matrix = largest row norm."** — Default `t.linalg.norm(z)` on
  2-D is FROBENIUS: all entries squared, summed, rooted — as if the matrix
  were one long vector. Operator norms exist behind `ord=`, but Frobenius is
  the drills' default meaning of "the norm".


<!-- dd:dd-kp-numpy-stack-concat-interleave -->

## Stacking, concatenating, interleaving

`numpy.stack-concat-interleave`


Combining arrays into one splits on a single question: **does the result have
a NEW axis, or grow an EXISTING one?**

- **Grow an existing dimension — `t.cat` and its 2-D shorthands.**
  `t.vstack([a, b])` stacks rows (b's rows below a's);
  `t.hstack([a, b])` extends rows sideways. Shapes must agree on the other
  axis; the combined axis just adds up. General form:
  `t.cat([a, b], dim=k)` — the shorthands are that call with `k` fixed.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\npair = [t.tensor([[1, 2]]), t.tensor([[3, 4]])]\nprint("dim=0 (taller):", t.cat(pair, dim=0).shape, t.cat(pair, dim=0).tolist())\nprint("dim=1 (wider): ", t.cat(pair, dim=1).shape, t.cat(pair, dim=1).tolist())\nprint("vstack is dim=0:", t.equal(t.vstack(pair), t.cat(pair, dim=0)))\n# Hidden checks\nassert _delta_output == \'dim=0 (taller): torch.Size([2, 2]) [[1, 2], [3, 4]]\\ndim=1 (wider):  torch.Size([1, 4]) [[1, 2, 3, 4]]\\nvstack is dim=0: True\\n\'\n', globals()), end='')



- **Create a new axis — `t.stack`.**
  `t.stack([a, b], axis=0)` piles k same-shape arrays into a (k, …) array.
  Nothing merges; you gain a dimension. This is the bridge to *reductions
  over the pile*: the elementwise average of two arrays is
  `t.stack([a, b]).mean(axis=0)` — stack, then reduce the new axis. Any
  "combine k arrays by taking the elementwise mean/max/median" is this
  two-step.
- **Interleave — stack + reshape, or strided assignment.**
  Alternating elements `[a0, b0, a1, b1, …]` has two idiomatic spellings:
  - `t.column_stack((a, b)).ravel()` — pair up (each row `[a_i, b_i]`),
    then read row-major: the pairs unroll in exactly alternating order.
    (Reshape's fill order doing real work!)
  - Preallocate and stride: `out[0::2] = a; out[1::2] = b` — allocate the
    full-length result, then write each source into its residue class.
    Generalizes cleanly to 3+ sources (`0::3`, `1::3`, `2::3`) and to
    "insert nz zeros between entries" (`out[::nz+1] = z` into a zeros
    canvas).

Choosing: piles that keep identity → stack; seams along an axis → concat
family; alternating patterns → column_stack+ravel or strided slots.


Task: stack two matrices vertically and horizontally; average them
elementwise via a stack; interleave two vectors.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\na = t.tensor([[1.0, 2.0],\n              [3.0, 4.0]])\nb = t.tensor([[5.0, 6.0],\n              [7.0, 8.0]])\n\n# Grow axis 0 (rows below) / axis 1 (columns to the right).\nv = t.vstack([a, b])\nh = t.hstack([a, b])\n\n# NEW axis then reduce it: elementwise average of the two arrays.\npiled = t.stack([a, b], dim=0)        # shape (2, 2, 2) — nothing merged\navg = piled.mean(dim=0)                # collapse the pile\n\n# Interleave two vectors: pair rows, then row-major ravel unrolls\n# them alternately.\nx = t.tensor([1, 3, 5])\ny = t.tensor([2, 4, 6])\ninter = t.column_stack((x, y)).ravel()\n\n# Same result by strided assignment — the form that scales to 3+ streams.\nout = t.empty(6, dtype=x.dtype)\nout[0::2] = x\nout[1::2] = y\nprint("vstack", tuple(v.shape), "| hstack", tuple(h.shape),\n      "| stack", tuple(piled.shape), "<- stack ADDS an axis")\nprint("averaged over the new axis:")\nprint(avg)\nprint("interleaved:", inter, "| by strided assignment:", out)\n# Hidden checks\nassert v.shape == (4, 2) and h.shape == (2, 4)\nassert piled.shape == (2, 2, 2)\nassert avg.tolist() == [[3.0, 4.0], [5.0, 6.0]]\nassert inter.tolist() == [1, 2, 3, 4, 5, 6]\nassert out.tolist() == [1, 2, 3, 4, 5, 6]\n', globals()), end='')




Why each step:

1. Track shapes: vstack (2,2)+(2,2)→(4,2) grew an axis; stack →(2,2,2) added
   one. The shape arithmetic is the reliable way to tell which operation a
   task describes.
2. stack-then-reduce turns "elementwise average/max of k arrays" into the
   axis machinery you already own — no dedicated function needed, and it
   generalizes from mean to any reduction.
3. Both interleave spellings matter: column_stack+ravel is elegant for two
   streams; residue-class assignment (`empty` first — safe here because
   every slot gets written) reads mechanically but handles any number of
   streams and irregular spacings.


<!-- dd:dd-q84 -->

### Problem 84 · faded — your turn

Elementwise average of two same-shape arrays, via a new axis.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2., 4.]])
```


In [ ]:
import torch as t

def solve(a, b):
    """Elementwise average: stack on a new axis, then reduce it."""
    return t._____([a, b], _____=0)._____(_____=0)


# Example run — the grader calls solve() with several pairs.
example_a = t.tensor([[1.0, 2.0]])
example_b = t.tensor([[3.0, 6.0]])
print(solve(example_a, example_b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(84)


In [ ]:
#@title 💡 Solution — Problem 84
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.stack([a, b], dim=0).mean(dim=0)


example_a = t.tensor([[1.0, 2.0]])
example_b = t.tensor([[3.0, 6.0]])
print(solve(example_a, example_b))


Task: stack two matrices vertically and horizontally; average them
elementwise via a stack; interleave two vectors.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\na = t.tensor([[1.0, 2.0],\n              [3.0, 4.0]])\nb = t.tensor([[5.0, 6.0],\n              [7.0, 8.0]])\n\n# Grow axis 0 (rows below) / axis 1 (columns to the right).\nv = t.vstack([a, b])\nh = t.hstack([a, b])\n\n# NEW axis then reduce it: elementwise average of the two arrays.\npiled = t.stack([a, b], dim=0)        # shape (2, 2, 2) — nothing merged\navg = piled.mean(dim=0)                # collapse the pile\n\n# Interleave two vectors: pair rows, then row-major ravel unrolls\n# them alternately.\nx = t.tensor([1, 3, 5])\ny = t.tensor([2, 4, 6])\ninter = t.column_stack((x, y)).ravel()\n\n# Same result by strided assignment — the form that scales to 3+ streams.\nout = t.empty(6, dtype=x.dtype)\nout[0::2] = x\nout[1::2] = y\nprint("vstack", tuple(v.shape), "| hstack", tuple(h.shape),\n      "| stack", tuple(piled.shape), "<- stack ADDS an axis")\nprint("averaged over the new axis:")\nprint(avg)\nprint("interleaved:", inter, "| by strided assignment:", out)\n# Hidden checks\nassert v.shape == (4, 2) and h.shape == (2, 4)\nassert piled.shape == (2, 2, 2)\nassert avg.tolist() == [[3.0, 4.0], [5.0, 6.0]]\nassert inter.tolist() == [1, 2, 3, 4, 5, 6]\nassert out.tolist() == [1, 2, 3, 4, 5, 6]\n', globals()), end='')




Why each step:

1. Track shapes: vstack (2,2)+(2,2)→(4,2) grew an axis; stack →(2,2,2) added
   one. The shape arithmetic is the reliable way to tell which operation a
   task describes.
2. stack-then-reduce turns "elementwise average/max of k arrays" into the
   axis machinery you already own — no dedicated function needed, and it
   generalizes from mean to any reduction.
3. Both interleave spellings matter: column_stack+ravel is elegant for two
   streams; residue-class assignment (`empty` first — safe here because
   every slot gets written) reads mechanically but handles any number of
   streams and irregular spacings.


<!-- dd:dd-q974 -->

### Problem 974 · faded — your turn

A seam along an axis that already exists: the row counts add up and the
column count is unchanged.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 2],
        [3, 4]])
```


In [ ]:
import torch as t


def solve(a, b):
    """The two matrices joined top to bottom."""
    return t._____([a, b], _____=0)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([[1, 2]]), t.tensor([[3, 4]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(974)


In [ ]:
#@title 💡 Solution — Problem 974
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(a, b):
    """The two matrices joined top to bottom."""
    return t.cat([a, b], dim=0)


print(solve(t.tensor([[1, 2]]), t.tensor([[3, 4]])))


Task: stack two matrices vertically and horizontally; average them
elementwise via a stack; interleave two vectors.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\na = t.tensor([[1.0, 2.0],\n              [3.0, 4.0]])\nb = t.tensor([[5.0, 6.0],\n              [7.0, 8.0]])\n\n# Grow axis 0 (rows below) / axis 1 (columns to the right).\nv = t.vstack([a, b])\nh = t.hstack([a, b])\n\n# NEW axis then reduce it: elementwise average of the two arrays.\npiled = t.stack([a, b], dim=0)        # shape (2, 2, 2) — nothing merged\navg = piled.mean(dim=0)                # collapse the pile\n\n# Interleave two vectors: pair rows, then row-major ravel unrolls\n# them alternately.\nx = t.tensor([1, 3, 5])\ny = t.tensor([2, 4, 6])\ninter = t.column_stack((x, y)).ravel()\n\n# Same result by strided assignment — the form that scales to 3+ streams.\nout = t.empty(6, dtype=x.dtype)\nout[0::2] = x\nout[1::2] = y\nprint("vstack", tuple(v.shape), "| hstack", tuple(h.shape),\n      "| stack", tuple(piled.shape), "<- stack ADDS an axis")\nprint("averaged over the new axis:")\nprint(avg)\nprint("interleaved:", inter, "| by strided assignment:", out)\n# Hidden checks\nassert v.shape == (4, 2) and h.shape == (2, 4)\nassert piled.shape == (2, 2, 2)\nassert avg.tolist() == [[3.0, 4.0], [5.0, 6.0]]\nassert inter.tolist() == [1, 2, 3, 4, 5, 6]\nassert out.tolist() == [1, 2, 3, 4, 5, 6]\n', globals()), end='')




Why each step:

1. Track shapes: vstack (2,2)+(2,2)→(4,2) grew an axis; stack →(2,2,2) added
   one. The shape arithmetic is the reliable way to tell which operation a
   task describes.
2. stack-then-reduce turns "elementwise average/max of k arrays" into the
   axis machinery you already own — no dedicated function needed, and it
   generalizes from mean to any reduction.
3. Both interleave spellings matter: column_stack+ravel is elegant for two
   streams; residue-class assignment (`empty` first — safe here because
   every slot gets written) reads mechanically but handles any number of
   streams and irregular spacings.


<!-- dd:dd-q975 -->

### Problem 975 · faded — your turn

The same kind of seam along the other axis, written with the 2-D shorthand
that fixes the axis for you.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 2, 3, 4]])
```


In [ ]:
import torch as t


def solve(a, b):
    """The two matrices joined side by side."""
    return t._____([a, b])


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([[1, 2]]), t.tensor([[3, 4]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(975)


In [ ]:
#@title 💡 Solution — Problem 975
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(a, b):
    """The two matrices joined side by side."""
    return t.hstack([a, b])


print(solve(t.tensor([[1, 2]]), t.tensor([[3, 4]])))


Task: stack two matrices vertically and horizontally; average them
elementwise via a stack; interleave two vectors.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\na = t.tensor([[1.0, 2.0],\n              [3.0, 4.0]])\nb = t.tensor([[5.0, 6.0],\n              [7.0, 8.0]])\n\n# Grow axis 0 (rows below) / axis 1 (columns to the right).\nv = t.vstack([a, b])\nh = t.hstack([a, b])\n\n# NEW axis then reduce it: elementwise average of the two arrays.\npiled = t.stack([a, b], dim=0)        # shape (2, 2, 2) — nothing merged\navg = piled.mean(dim=0)                # collapse the pile\n\n# Interleave two vectors: pair rows, then row-major ravel unrolls\n# them alternately.\nx = t.tensor([1, 3, 5])\ny = t.tensor([2, 4, 6])\ninter = t.column_stack((x, y)).ravel()\n\n# Same result by strided assignment — the form that scales to 3+ streams.\nout = t.empty(6, dtype=x.dtype)\nout[0::2] = x\nout[1::2] = y\nprint("vstack", tuple(v.shape), "| hstack", tuple(h.shape),\n      "| stack", tuple(piled.shape), "<- stack ADDS an axis")\nprint("averaged over the new axis:")\nprint(avg)\nprint("interleaved:", inter, "| by strided assignment:", out)\n# Hidden checks\nassert v.shape == (4, 2) and h.shape == (2, 4)\nassert piled.shape == (2, 2, 2)\nassert avg.tolist() == [[3.0, 4.0], [5.0, 6.0]]\nassert inter.tolist() == [1, 2, 3, 4, 5, 6]\nassert out.tolist() == [1, 2, 3, 4, 5, 6]\n', globals()), end='')




Why each step:

1. Track shapes: vstack (2,2)+(2,2)→(4,2) grew an axis; stack →(2,2,2) added
   one. The shape arithmetic is the reliable way to tell which operation a
   task describes.
2. stack-then-reduce turns "elementwise average/max of k arrays" into the
   axis machinery you already own — no dedicated function needed, and it
   generalizes from mean to any reduction.
3. Both interleave spellings matter: column_stack+ravel is elegant for two
   streams; residue-class assignment (`empty` first — safe here because
   every slot gets written) reads mechanically but handles any number of
   streams and irregular spacings.


<!-- dd:dd-q976 -->

### Problem 976 · faded — your turn

Pair-and-unroll: once each row holds one entry from each source, reading
row by row IS the alternating order.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 2, 3, 4, 5, 6])
```


In [ ]:
import torch as t


def solve(a, b):
    """The two vectors alternated, a first."""
    return t._____((a, b))._____()


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([1, 3, 5]), t.tensor([2, 4, 6])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(976)


In [ ]:
#@title 💡 Solution — Problem 976
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(a, b):
    """The two vectors alternated, a first."""
    return t.column_stack((a, b)).ravel()


print(solve(t.tensor([1, 3, 5]), t.tensor([2, 4, 6])))


<!-- dd:dd-q146 -->

### Problem 146 · guided

Write a function solve(a, b, c) that takes three 1-D PyTorch tensors of the same length n and returns a single tensor of length 3n interleaving them position by position: the result reads a[0], b[0], c[0], a[1], b[1], c[1], and so on. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([  1,  10, 100,   2,  20, 200])
```


<details>
<summary>Hints</summary>

1. Three streams interleaved position by position — the pair-and-ravel trick
   still works, but the strided form is clearer: what are the three residue
   classes?
2. Allocate the result (`t.empty(3 * n, dtype=...)` — dtype from the inputs
   via `t.result_type`), then one slice assignment per stream.
3. `out[0::3] = a; out[1::3] = b; out[2::3] = c`.

</details>


In [ ]:
import torch as t

def solve(a, b, c):
    """Return a, b, c interleaved elementwise into one length-3n array."""
    return None


# Example run — the grader calls solve() with several triples.
print(solve(t.tensor([1, 2]), t.tensor([10, 20]), t.tensor([100, 200])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(146)


In [ ]:
#@title 💡 Solution — Problem 146
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(a, b, c):
    dtype = t.promote_types(t.result_type(a, b), c.dtype)
    out = t.empty(a.numel() * 3, dtype=dtype)
    out[0::3] = a
    out[1::3] = b
    out[2::3] = c
    return out


print(solve(t.tensor([1, 2]), t.tensor([10, 20]), t.tensor([100, 200])))


<!-- dd:dd-q977 -->

### Problem 977 · guided

A Python list of k tensors, all the same shape, is given. Return a tuple holding the single tensor that piles them up along a brand-new FIRST axis, and that tensor's shape as a plain tuple of ints.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([[[1, 2]],

        [[3, 4]]]), (2, 1, 2))
```


<details>
<summary>Hints</summary>

1. The two joining tools differ on one question: does the result gain an axis, or does an existing one get longer? Here every input has to stay identifiable, so it gains one.
2. `t.stack` takes the list and the position of the new axis. Piling k tensors of shape (r, c) at position 0 gives (k, r, c).
3. `t.stack(mats, dim=0)`, then `tuple(...shape)` on the result.

</details>


In [ ]:
import torch as t


def solve(mats):
    """The pile of same-shape matrices, and its shape."""
    return  # your answer here

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve([t.tensor([[1, 2]]), t.tensor([[3, 4]])]))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(977)


In [ ]:
#@title 💡 Solution — Problem 977
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(mats):
    """The pile of same-shape matrices, and its shape."""
    piled = t.stack(mats, dim=0)
    return piled, tuple(piled.shape)


print(solve([t.tensor([[1, 2]]), t.tensor([[3, 4]])]))


<!-- dd:dd-q89 -->

### Problem 89 · independent

Write a function solve(a, b) that takes two 1-D PyTorch tensors of the same length n and returns a single 1-D tensor of length 2n whose entries alternate between a and b starting with a[0]: [a[0], b[0], a[1], b[1], ...]. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 2, 3, 4, 5, 6])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return a and b interleaved: a[0], b[0], a[1], b[1], ..."""
    return None


# Example run — the grader calls solve() with several pairs.
print(solve(t.tensor([1, 3, 5]), t.tensor([2, 4, 6])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(89)


In [ ]:
#@title 💡 Solution — Problem 89
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.ravel(t.column_stack((a, b)))


print(solve(t.tensor([1, 3, 5]), t.tensor([2, 4, 6])))


<!-- dd:dd-q238 -->

### Problem 238 · independent

Write a function solve(a, b) that takes two 2-D PyTorch tensors a and b with the same shape. It should return a tuple of two tensors: the first is the vertical combination, with the rows of b stacked below the rows of a; the second is the horizontal combination, with b placed side by side to the right of a. Do not modify the input tensors.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([[1., 2.],
        [3., 4.],
        [5., 6.],
        [7., 8.]]), tensor([[1., 2., 5., 6.],
        [3., 4., 7., 8.]]))
```


In [ ]:
import torch as t

def solve(a, b):
    """Return (vertical combination of a and b, horizontal combination of a and b)."""
    return None


# Example run — the grader calls solve() with several different array pairs,
# including edge cases. Your function must work for all of them.
a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(a, b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(238)


In [ ]:
#@title 💡 Solution — Problem 238
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.vstack([a, b]), t.hstack([a, b])


a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(a, b))


<!-- dd:dd-q159 -->

### Problem 159 · independent

Write a function solve(z, nz) that takes a 1-D PyTorch tensor z and a non-negative integer nz, and returns a new tensor in which nz zeros are interleaved between every pair of consecutive entries of z: the result has length len(z) + (len(z) - 1) * nz, with z's values at positions 0, nz+1, 2*(nz+1), ...

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 0., 0., 0., 2., 0., 0., 0., 3., 0., 0., 0., 4., 0., 0., 0., 5.])
```


In [ ]:
import torch as t

def solve(z, nz):
    """Return z with nz zeros inserted between consecutive entries."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([1, 2, 3, 4, 5]), 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(159)


In [ ]:
#@title 💡 Solution — Problem 159
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, nz):
    out = t.zeros(len(z) + (len(z) - 1) * nz)
    out[:: nz + 1] = z
    return out


print(solve(t.tensor([1, 2, 3, 4, 5]), 3))


<!-- dd:dd-q978 -->

### Problem 978 · independent

A Python list of tensors, all the same shape, is given. Return the tensor whose every position holds the largest value found at that position across the whole list. No loop.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([4, 5])
```


In [ ]:
import torch as t


def solve(mats):
    """The elementwise maximum across a list of same-shape tensors."""
    return  # your answer here

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve([t.tensor([1, 5]), t.tensor([4, 2])]))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(978)


In [ ]:
#@title 💡 Solution — Problem 978
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(mats):
    """The elementwise maximum across a list of same-shape tensors."""
    return t.stack(mats, dim=0).max(dim=0).values


print(solve([t.tensor([1, 5]), t.tensor([4, 2])]))


<!-- dd:dd-q979 -->

### Problem 979 · independent

Three 1-D integer tensors of the same length n are given. Return the 1-D tensor of length three times n that takes one entry from each in turn: a's first, b's first, c's first, then a's second, and so on. No loop.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 2, 3, 4, 5, 6])
```


In [ ]:
import torch as t


def solve(a, b, c):
    """Three vectors interleaved, a then b then c at each position."""
    return  # your answer here

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([1, 4]), t.tensor([2, 5]), t.tensor([3, 6])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(979)


In [ ]:
#@title 💡 Solution — Problem 979
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(a, b, c):
    """Three vectors interleaved, a then b then c at each position."""
    return t.column_stack((a, b, c)).ravel()


print(solve(t.tensor([1, 4]), t.tensor([2, 5]), t.tensor([3, 6])))


<!-- dd:dd-q980 -->

### Problem 980 · independent

A 2-D tensor is given along with a single integer. Return the tensor with one extra column of that constant value added on the LEFT and another on the RIGHT. The row count is unchanged and the column count grows by two.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2, 0],
        [0, 3, 4, 0]])
```


In [ ]:
import torch as t


def solve(z, pad):
    """The matrix with one constant column added on each side."""
    return  # your answer here

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
print(solve(t.tensor([[1, 2], [3, 4]]), 0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(980)


In [ ]:
#@title 💡 Solution — Problem 980
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(z, pad):
    """The matrix with one constant column added on each side."""
    side = t.full((z.shape[0], 1), pad, dtype=z.dtype)
    return t.cat([side, z, side], dim=1)


print(solve(t.tensor([[1, 2], [3, 4]]), 0))


#### Common mistakes

- **"stack and concatenate are synonyms."** — concatenate grows an existing
  axis (no new dimension); stack creates a new one. (2,3)+(2,3): concat
  axis-0 → (4,3); stack → (2,2,3). The task's result shape tells you which.
- **"Interleaving needs a Python loop."** — Either pair-and-ravel
  (column_stack + row-major flatten) or strided slice assignment. Both are
  single-pass, loop-free.
- **"t.empty is dangerous here."** — It's uninitialized memory, which is
  fine EXACTLY when every slot gets written before any read — as in the
  residue-class pattern. If any slot might stay untouched (the zeros-between
  drill!), start from `t.zeros` instead.
